# テキストデータの重複と不良データを取り除く

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/01_data_io/dedup_clean.ipynb)

NLP クックブック 01-07 ／ データの収集と入出力

集めたテキストに、完全な重複、空白や大文字小文字だけが違う重複、短すぎる・長すぎる行、文字化けが混ざっている。学習前に掃除したい。

In [1]:
import re
import unicodedata
import pandas as pd

raw = ["今日は良い天気です。", "今日は良い天気です。", "今日は　良い天気です。 ", "ＮＬＰは楽しい", "NLPは楽しい", "", "あ", "!!!!!!!!!!!!",
       "縺薙ｓ縺ｫ縺｡縺ｯ荳也阜", "https://example.com/only-url", "機械翻訳の品質は近年大きく向上した。", "This line is English only.",
       "長い文" * 300]
df = pd.DataFrame({"text": raw})

def normalize_key(s):
    """重複判定用のキー: 正規化して空白と記号を除く (原文は書き換えない)"""
    s = unicodedata.normalize("NFKC", s).lower()
    return re.sub(r"[\s\W_]+", "", s)

def ja_ratio(s):
    return len(re.findall(r"[ぁ-んァ-ヶ一-龥]", s)) / max(1, len(s))

def looks_mojibake(s):
    return len(re.findall(r"[縺繧繝荳螟譁蜿]", s)) / max(1, len(s)) > 0.2   # UTF-8 を Shift_JIS と誤解したときに頻出する文字

df["key"] = df["text"].map(normalize_key)
checks = {
    "空・短すぎる": df["key"].str.len() < 3,
    "長すぎる": df["text"].str.len() > 500,
    "日本語が少ない": df["text"].map(ja_ratio) < 0.3,
    "文字化け": df["text"].map(looks_mojibake),
    "重複": df.duplicated("key", keep="first"),
}
df["reason"] = ""
for name, mask in checks.items():
    df.loc[mask & (df["reason"] == ""), "reason"] = name
print(df["reason"].replace("", "(残す)").value_counts())
df[df["reason"] == ""]["text"].tolist()

reason
(残す)       3
重複         3
空・短すぎる     3
日本語が少ない    2
文字化け       1
長すぎる       1
Name: count, dtype: int64


['今日は良い天気です。', 'ＮＬＰは楽しい', '機械翻訳の品質は近年大きく向上した。']

- 何を何件捨てたかを理由ごとに記録しておく。フィルタのバグでデータの大半を捨てていた、という事故に気づける。
- 学習データと評価データの間の重複（リーク）も必ず確認する: `set(train_keys) & set(test_keys)`。
- ほぼ同じ内容の文書（ニアデュプリケート）は、この章の「MinHash と LSH で大量文書からほぼ重複を見つける」で検出する。